# Phonetic ASR Baseline and SLM Dataset Preparation

This notebook analyzes the baseline ASR behavior on the cleaned 50-word child speech dataset and prepares binary classification examples for MiniCPM fine-tuning.

The baseline ASR metric is intentionally strict: the transcript must exactly match the target word after case normalization. That strict score is useful as a pre-fine-tuning reference, but it over-penalizes phonetic near-matches such as pluralization, word splitting, or plausible lisp-driven ASR substitutions. For SLM training, we transform each row into a binary judgment task: decide whether the ASR transcript is a valid phonetic match for the target word.

In [6]:
import json
from pathlib import Path

import pandas as pd

baseline_path = Path("data/baseline_results.csv")
if not baseline_path.exists():
    baseline_path = Path("../data/baseline_results.csv")
df = pd.read_csv(baseline_path)

# Normalize CSV booleans in case the file is edited or re-exported by a spreadsheet tool.
df["Strict Match"] = df["Strict Match"].astype(str).str.lower().eq("true")

accuracy = df["Strict Match"].mean()
print(f"Rows: {len(df)}")
print(f"Strict baseline accuracy: {accuracy:.2%} ({df['Strict Match'].sum()}/{len(df)})")
df.head()

Rows: 50
Strict baseline accuracy: 76.00% (38/50)


,File,Target Word,ASR Transcript,Strict Match
0,data/processed_audio/jane_cleaned/01_scientist...,scientist,scientists,False
1,data/processed_audio/jane_cleaned/02_safari.wav,safari,so far,False
2,data/processed_audio/jane_cleaned/03_secret.wav,secret,secret,True
3,data/processed_audio/jane_cleaned/04_sunflower...,sunflower,sunny flowers,False
4,data/processed_audio/jane_cleaned/05_system.wav,system,system,True


In [7]:
failures = df[df["Strict Match"] == False].copy()
failures.head(20)

,File,Target Word,ASR Transcript,Strict Match
0,data/processed_audio/jane_cleaned/01_scientist...,scientist,scientists,False
1,data/processed_audio/jane_cleaned/02_safari.wav,safari,so far,False
3,data/processed_audio/jane_cleaned/04_sunflower...,sunflower,sunny flowers,False
7,data/processed_audio/jane_cleaned/08_sudden.wav,sudden,seven,False
11,data/processed_audio/jane_cleaned/12_invisible...,invisible,and where's the ball,False
12,data/processed_audio/jane_cleaned/13_recipe.wav,recipe,this is tea,False
13,data/processed_audio/jane_cleaned/14_fossil.wav,fossil,five,False
15,data/processed_audio/jane_cleaned/16_massive.wav,massive,maze,False
17,data/processed_audio/jane_cleaned/18_guessing.wav,guessing,yeah same,False
28,data/processed_audio/jane_cleaned/29_compass.wav,compass,come this,False


## Strict ASR Failure Examples

| Target Word | ASR Transcript | Notes |
| --- | --- | --- |
| scientist | scientists | Pluralized but phonetically close |
| safari | so far | Slow/unclear pronunciation split into common words |
| sunflower | sunny flowers | Compound split and inflected |
| sudden | seven | Different word |
| invisible | and where's the ball | Unrelated speech hallucination |
| recipe | this is tea | Unrelated phrase |
| fossil | five | Different word |
| massive | maze | Insufficient phonetic evidence |
| guessing | yeah same | Unrelated phrase |
| compass | come this | Plausible lisp/ASR rendering |
| pyramid | apparently | Different word |
| window | wind up | Close ASR phrase boundary |

This table is intentionally included as rendered markdown so the baseline failure modes are visible even before executing the notebook.

## Binary Classification Transformation

The fine-tuning task is framed as a compact binary classifier. Given a target word and an ASR transcript, the model should output only `True` or `False`.

- Strict ASR matches automatically become `True` examples.
- Strict failures are manually labeled because some are still acceptable phonetic matches. For example, `scientist -> scientists` preserves the target pronunciation plus a plural ending, while `invisible -> and where's the ball` is unrelated speech.
- The resulting JSONL is deliberately simple so the SLM learns the decision boundary rather than a verbose explanation style.

In [8]:
failure_output_map = {
    # Plausible phonetic/ASR variants that should count as valid readings.
    "scientist": "True",      # pluralized ASR output: scientists
    "safari": "True",         # ASR split a slow pronunciation into: so far
    "sunflower": "True",      # ASR split/inflected the compound: sunny flowers
    "compass": "True",        # plausible lisp/ASR rendering: come this
    "window": "True",         # close ASR phrase boundary: wind up

    # Wrong-content hallucinations or insufficient phonetic evidence.
    "sudden": "False",
    "invisible": "False",
    "recipe": "False",
    "fossil": "False",
    "massive": "False",
    "guessing": "False",
    "pyramid": "False",
}

failure_rationale = pd.DataFrame(
    [
        {
            "Target Word": row["Target Word"],
            "ASR Transcript": row["ASR Transcript"],
            "Mapped Output": failure_output_map[row["Target Word"]],
        }
        for _, row in failures.iterrows()
    ]
)
failure_rationale

,Target Word,ASR Transcript,Mapped Output
0,scientist,scientists,True
1,safari,so far,True
2,sunflower,sunny flowers,True
3,sudden,seven,False
4,invisible,and where's the ball,False
5,recipe,this is tea,False
6,fossil,five,False
7,massive,maze,False
8,guessing,yeah same,False
9,compass,come this,True


In [9]:
instruction = "Determine if the ASR transcript is a valid phonetic match for the target word. Output only True or False."
examples = []

for _, row in df.iterrows():
    target_word = str(row["Target Word"]).strip()
    transcript = str(row["ASR Transcript"]).strip()

    if row["Strict Match"]:
        output = "True"
    else:
        if target_word not in failure_output_map:
            raise KeyError(f"Missing manual label for failed target word: {target_word}")
        output = failure_output_map[target_word]

    examples.append(
        {
            "instruction": instruction,
            "input": f"Target: {target_word} | ASR: {transcript}",
            "output": output,
        }
    )

print(f"Prepared {len(examples)} training examples")
print(f"Positive examples: {sum(example['output'] == 'True' for example in examples)}")
print(f"Negative examples: {sum(example['output'] == 'False' for example in examples)}")
examples[:5]

Prepared 50 training examples
Positive examples: 43
Negative examples: 7


[{'instruction': 'Determine if the ASR transcript is a valid phonetic match for the target word. Output only True or False.',
  'input': 'Target: scientist | ASR: scientists',
  'output': 'True'},
 {'instruction': 'Determine if the ASR transcript is a valid phonetic match for the target word. Output only True or False.',
  'input': 'Target: safari | ASR: so far',
  'output': 'True'},
 {'instruction': 'Determine if the ASR transcript is a valid phonetic match for the target word. Output only True or False.',
  'input': 'Target: secret | ASR: secret',
  'output': 'True'},
 {'instruction': 'Determine if the ASR transcript is a valid phonetic match for the target word. Output only True or False.',
  'input': 'Target: sunflower | ASR: sunny flowers',
  'output': 'True'},
 {'instruction': 'Determine if the ASR transcript is a valid phonetic match for the target word. Output only True or False.',
  'input': 'Target: system | ASR: system',
  'output': 'True'}]

In [10]:
output_path = Path("data/train.jsonl")
if not output_path.parent.exists():
    output_path = Path("../data/train.jsonl")

with output_path.open("w", encoding="utf-8") as handle:
    for example in examples:
        handle.write(json.dumps(example, ensure_ascii=False) + "\n")

print(f"Wrote {len(examples)} examples to {output_path}")

Wrote 50 examples to ../data/train.jsonl
